In [1]:
import pandas as pd
import re
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
# Do not show FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings(
    'ignore',
    category=UserWarning,
    message=(
        "Creating legend with loc=\"best\" can be slow"
        " with large amounts of data."
    )
)

from target_extractor.graph_gen import Graph

pd.set_option('display.max_columns', None)

In [2]:
# Import the dataframe with masterfile data that contains both
# outliers and normal compounds (inliers)
master_file = ("../data/masterfile.xlsx")
df_master = pd.read_excel(master_file, sheet_name="Sheet1")

# Fill NaN values in the "Screen: Effect on EV uptake" column
# with "Normal" for better clarity in analysis
df_master["Screen: Effect on EV uptake"] = (
    df_master["Screen: Effect on EV uptake"].fillna("Normal")
)


In [3]:
df_master.head()

,Number,Library,Screen: Effect on EV uptake,Screen: EV-uptake_Normalized_by_mean,Compound Name,Target UniProt ID,Target Abbreviation,Target Name,Mechanism of Action,Keywords,Effect on Target protein (Literature),Binding Site (co-pilot),Effect (co-pilot),Literature reference,specific_targets,extra_explanation,extra_ref
0,205.0,Prestwick,Inducer,0.010239,"Antipyrine, 4-hydroxy","P23219, P35354",NaN,"Cytochrome P450 2C9, Prostaglandin G/H synthas...",This is a metabolite of Antipyrine (Phenazone)...,Cytochrome; Prostaglandin,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,238.0,Prestwick,Inducer,0.010709,"Homatropine hydrobromide (R,S)",P11229,NaN,Muscarinic acetylcholine receptor M1,Homatropine is a non-selective muscarinic rece...,Acetylcholine receptor,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Normal,NaN,(-)-alpha-Methylnorepinephrine,P08100,ADRA1A,Alpha-1A adrenergic receptor,"Direct agonist of ADRA1A, a major vascular alp...",adrenergic receptor; Na+/K+-ATPase,NaN,NaN,NaN,NaN,Sodium-potas-ATPase,Stimulates Na+/K+-ATPase activity in renal cel...,https://pubmed.ncbi.nlm.nih.gov/10737614/
3,NaN,NaN,Normal,NaN,"(-)-cis-(1S,2R)-U-50488 tartrate",P41145,OPRK1,Kappa-type opioid receptor,"Selective agonist for OPRK1, the kappa opioid ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Normal,NaN,(-)-Cotinine,P08172,CHRNA4,Neuronal acetylcholine receptor subunit alpha-4,"Directly binds nicotinic receptors, especially...",Acetylcholine receptor,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# unique libraries in the "Library" column
unique_libraries = df_master["Library"].unique()
unique_libraries

array(['Prestwick', nan, 'LOPAC'], dtype=object)

# Defining essential custom functions 

In [5]:
g = Graph(df_master, type_col="Screen: Effect on EV uptake")
fig = g.scored_bar_plot()

df rows: 2106


In [ ]:
fig.write_image("../results/prevalence_targets_graph.svg")

In [ ]:
# export the graph as html file
# fig.write_html("../results/prevalence_targets_graph.html")

In [6]:
g = Graph(df_master, type_col="Screen: Effect on EV uptake")
df_table = g.scored_table()

In [7]:
df_table.head()

,Acetylcholine receptor,Adenosine deaminase,adrenergic receptor,Aldehyde dehydrogenase,angiotensin,carbonic anhydrase,cyclic phosphodiesterase,Cytochrome,Dihydropteroate synthase,DNA topoisomerase,Dopamine,Glucocorticoid,Glutamate,Histamine,HMG-CoA reductase,Hydroxytryptamine,Lanosterol,lipoxygenase,Na+/K+-ATPase,Progesteron,Prostaglandin,protein synthesis,Sodium channel,Tubulin,Vitamin,Voltage-dependent L-type calcium channel,YAP/TAZ-TEAD
0,Camylofine chlorhydrate,Cladribine,RX 821002 hydrochloride,Disulfiram,Perindopril,Ethoxzolamide,Irsogladine maleate,Metyrapone,4-aminosalicylic acid,Daunorubicin hydrochloride,Fenoldopam bromide,Flunisolide,SDZ 220-581 hydrochloride,Dimenhydrinate,Atorvastatin,Methiothepin mesylate,Oxiconazole Nitrate,Zileuton,Fenoldopam bromide,Ethynodiol diacetate,Nabumetone,Sisomicin sulfate,Dyclonine hydrochloride,Oxfendazol,Acenocoumarol,Isradipine,Nifuroxazide
1,Tropisetron hydrochloride,Fludarabine,Buflomedil hydrochloride,None,None,None,Vardenafil,"Antipyrine, 4-hydroxy",Sulfamethizole,Doxorubicin hydrochloride,Bromperidol,Cortisol acetate,Acamprosate calcium,Dibenzepine hydrochloride,Pitavastatin calcium,Tropisetron hydrochloride,Terconazole,Atreleuton,Aminoguanidine hydrochloride,None,Celecoxib,Lincomycin hydrochloride,Bupivacaine hydrochloride,Griseofulvin,Alfacalcidol,Nisoldipine,Linagliptin
2,Dimethisoquin hydrochloride,None,Ritodrine hydrochloride,None,None,None,Aminophylline,Ampyrone,Sulfamonomethoxine,Norfloxacin,Ziprasidone Hydrochloride,Flurandrenolide,Amantadine fumarate,Roxatidine Acetate hydrochloride,Fluvastatin sodium salt,Dibenzepine hydrochloride,Butoconazole nitrate,None,Doxorubicin hydrochloride,None,Isopyrin hydrochloride,Thiamphenicol,Oxethazaine,Vincristine sulfate,Nicotinamide,Suloctidil,Disulfiram
3,Oxolamine citrate salt,None,Isometheptene mucate,None,None,None,Ethaverine hydrochloride,None,Sulfachloropyridazine,Ciprofloxacin hydrochloride monohydrate,Octoclothepin maleate salt,Prednisolone,Avermectin B1,Chlorcyclizine hydrochloride,None,Ziprasidone Hydrochloride,Sulconazole nitrate,None,Sibutramine hydrochloride,None,Etoricoxib,Lymecycline,Diperodon hydrochloride,CID 11210285 hydrochloride,None,Amlodipine,Atorvastatin
4,"Homatropine hydrobromide (R,S)",None,Phenoxybenzamine hydrochloride,None,None,None,None,None,Sulfadoxine,Epirubicin hydrochloride,Zotepine,Fluticasone propionate,D(-)-2-Amino-5-phosphonopentanoic acid,Azelastine hydrochloride,None,Octoclothepin maleate salt,Posaconazole,None,Diethylstilbestrol,None,"Antipyrine, 4-hydroxy",None,Amiodarone hydrochloride,Podophyllotoxin,None,Amiodarone hydrochloride,Amlodipine


In [ ]:
# import tested_outliers.xlsx from data folder
outliers = pd.read_excel("../data/tested_outliers.xlsx")
# extract the name of the compounds as a list from the "Compounds" column
outlier_compounds = outliers["Compounds"].tolist()

table =g.compound_finder(outlier_compounds)
table.head(100)

In [ ]:
# export pandas dataframe as excel file
# table.to_excel("../results/bar_graph_table.xlsx", index=False)

# -----------------------

In [3]:
g = Graph(df_master, type_col="Screen: Effect on EV uptake", height=1000)
fig = g.bar_plot()

df rows: 2106


# Make separate bargraphs for outliers and inliers


In [10]:
# Make a seprate df for normal compounds only
df_normal_comp = df_master[
    df_master["Screen: Effect on EV uptake"] == "Normal"
].copy()

# Make a seprate df for outliers only
df_outliers = df_master[
    df_master["Screen: Effect on EV uptake"].isin(["Inhibitor", "Inducer"])
].copy()

In [11]:
fig = g.outliers_plot(
df_outliers,
df_normal_comp,
outlier_label="Outliers", # optional – changes subplot title & print label
normal_label="Normal compounds", # optional
outlier_color="steelblue", # optional – defaults already match original
normal_color="indianred", # optional
title="Keyword Frequency (alphabetical x-axis)", # optional
)

Outliers rows: 243
Normal compounds rows: 1863
